# WorldStrat LR / HR Viewer Notebook

For Super-Resolution tasks. Uses the improved visualization from previous scripts.

In [ ]:
import rasterio
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import os
from tqdm import tqdm

# ================== SINGLE LR VIEWER ==================
def view_lr_tiff(tiff_path):
    with rasterio.open(tiff_path) as src:
        print("=== LR Metadata ===")
        print(src.profile)
        data = src.read()
        print("\n=== LR Band Statistics ===")
        for i in range(src.count):
            band = data[i]
            print(f"Band {i+1}: min={band.min():.4f}, max={band.max():.4f}, mean={band.mean():.4f}")
        
        # Sentinel-2 typical RGB: B4, B3, B2 (0-based indices 3,2,1)
        red = data[3].astype(np.float32)
        green = data[2].astype(np.float32)
        blue = data[1].astype(np.float32)
        
        def stretch(band):
            p2, p98 = np.percentile(band, (2, 98))
            return np.clip((band - p2) / (p98 - p2 + 1e-6) * 255, 0, 255)
        
        rgb = np.dstack((stretch(red), stretch(green), stretch(blue))).astype(np.uint8)
        
        plt.figure(figsize=(10, 10))
        plt.imshow(rgb)
        plt.title(f"LR: {os.path.basename(tiff_path)}")
        plt.axis('off')
        plt.show()
        
        save_path = tiff_path.replace('.tiff', '_visualized.png')
        Image.fromarray(rgb).save(save_path)
        print(f"Saved: {save_path}")
        return rgb

In [ ]:
# ================== SINGLE HR VIEWER ==================
def view_hr_tiff(tiff_path):
    with rasterio.open(tiff_path) as src:
        print("=== HR Metadata ===")
        print(src.profile)
        data = src.read()
        print("\n=== HR Band Statistics ===")
        for i in range(src.count):
            band = data[i]
            print(f"Band {i+1}: min={band.min():.2f}, max={band.max():.2f}, mean={band.mean():.2f}")
        
        # SPOT ps.tiff correct ordering: Band1=Blue, Band2=Green, Band3=Red
        blue = data[0].astype(np.float32)
        green = data[1].astype(np.float32)
        red = data[2].astype(np.float32)
        
        def stretch(band):
            p2, p98 = np.percentile(band[band > 0], (2, 98)) if np.any(band > 0) else (0, 255)
            return np.clip((band - p2) / (p98 - p2 + 1e-6) * 255, 0, 255)
        
        rgb = np.dstack((stretch(red), stretch(green), stretch(blue))).astype(np.uint8)
        
        plt.figure(figsize=(10, 10))
        plt.imshow(rgb)
        plt.title(f"HR: {os.path.basename(tiff_path)}")
        plt.axis('off')
        plt.show()
        
        save_path = tiff_path.replace('.tiff', '_visualized.png')
        Image.fromarray(rgb).save(save_path)
        print(f"Saved: {save_path}")
        return rgb

In [ ]:
# ================== SIDE-BY-SIDE COMPARISON ==================
def compare_lr_hr(lr_path, hr_path):
    lr_rgb = view_lr_tiff(lr_path)  # This will also print stats and show image
    hr_rgb = view_hr_tiff(hr_path)
    
    fig, axs = plt.subplots(1, 2, figsize=(20, 10))
    axs[0].imshow(lr_rgb)
    axs[0].set_title('LR Image')
    axs[0].axis('off')
    axs[1].imshow(hr_rgb)
    axs[1].set_title('HR Image')
    axs[1].axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
# ================== BATCH PROCESS LR ==================
def batch_visualize_lr(input_dir, output_base_dir='/home/hassan/Documents/Hassan/SR Enhancement Work/Datasets/WorldStrat/visualized_worldstrat/visualized_lr'):
    os.makedirs(output_base_dir, exist_ok=True)
    
    # Collect files first for clean tqdm progress
    lr_files = []
    for root, dirs, files in os.walk(input_dir):
        for file in files:
            if file.endswith(('-L2A_data.tiff', '.tiff')) and 'L2A_data' in file:
                lr_files.append((root, file))
    
    for root, file in tqdm(lr_files, desc="Processing LR images", unit="image"):
        tiff_path = os.path.join(root, file)
        rel_path = os.path.relpath(root, input_dir)
        out_dir = os.path.join(output_base_dir, rel_path)
        os.makedirs(out_dir, exist_ok=True)
        
        # view_lr_tiff will print stats and save the visualized PNG
        view_lr_tiff(tiff_path)

In [ ]:
# ================== BATCH PROCESS HR ==================
def batch_visualize_hr(input_dir, output_base_dir='/home/hassan/Documents/Hassan/SR Enhancement Work/Datasets/WorldStrat/visualized_worldstrat/visualized_hr'):
    os.makedirs(output_base_dir, exist_ok=True)
    
    # Collect files first for clean tqdm progress
    hr_files = []
    for root, dirs, files in os.walk(input_dir):
        for file in files:
            if file.endswith(('-ps.tiff', '-pan.tiff')):
                hr_files.append((root, file))
    
    for root, file in tqdm(hr_files, desc="Processing HR images", unit="image"):
        tiff_path = os.path.join(root, file)
        rel_path = os.path.relpath(root, input_dir)
        out_dir = os.path.join(output_base_dir, rel_path)
        os.makedirs(out_dir, exist_ok=True)
        
        # view_hr_tiff will print stats and save the visualized PNG
        view_hr_tiff(tiff_path)

## Usage Examples

```python
# Single LR
lr_path = 'path/to/Amnesty POI-1-1-1-1-L2A_data.tiff'
view_lr_tiff(lr_path)

# Single HR
hr_path = 'path/to/Amnesty POI-2-3-2_ps.tiff'
view_hr_tiff(hr_path)

# Side by side
compare_lr_hr(lr_path, hr_path)

# Batch (uses your preferred output paths)
batch_visualize_lr('/path/to/lr_dataset')
batch_visualize_hr('/path/to/hr_dataset')
```